# 02 · Superficie de respuesta de segundo orden — CCD (Python)

**Semana 4 — Optimización (RSM).**

## Objetivos
- Ajustar un **modelo cuadrático** con un **diseño central compuesto** (CCD).
- Localizar el **punto óptimo**, clasificarlo (análisis canónico) y graficar contornos.

> Teoría: [`../teoria/02-segundo-orden-ccd-bb.md`](../teoria/02-segundo-orden-ccd-bb.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

np.random.seed(8)

## 1. El diseño CCD

Cerca del óptimo. Dos factores codificados $x_1,x_2$: 4 puntos **factoriales** ($\pm1$), 4 **axiales** ($\pm1.414$, rotable) y 5 **centrales**. Respuesta = rendimiento (%).

In [ ]:
df = pd.read_csv('../datos/rsm-ccd.csv')
df

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(df.x1, df.x2, s=60)
ax.axhline(0, color='gray', lw=.5); ax.axvline(0, color='gray', lw=.5)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Puntos del CCD (factoriales, axiales, centrales)'); plt.show()

## 2. Modelo de segundo orden

$\hat{y}=\beta_0+\beta_1x_1+\beta_2x_2+\beta_{11}x_1^2+\beta_{22}x_2^2+\beta_{12}x_1x_2$.

In [ ]:
modelo = smf.ols('rendimiento ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2', data=df).fit()
print(modelo.params.round(3))
print(f'R2 = {modelo.rsquared:.4f}')

## 3. Punto estacionario y análisis canónico

El óptimo resuelve $\nabla\hat{y}=0$: $\mathbf{x}_s=-\tfrac12\mathbf{B}^{-1}\mathbf{b}$. Los **valores propios** de $\mathbf{B}$ dicen si es máximo (todos $<0$), mínimo (todos $>0$) o silla (signos mixtos).

In [ ]:
p = modelo.params
b = np.array([p['x1'], p['x2']])
B = np.array([[p['I(x1 ** 2)'], p['x1:x2']/2],
              [p['x1:x2']/2, p['I(x2 ** 2)']]])
xs = -0.5 * np.linalg.solve(B, b)
ev = np.linalg.eigvals(B)
print(f'Punto estacionario: x1={xs[0]:.3f}, x2={xs[1]:.3f}')
print(f'Valores propios: {ev.round(3)}')
tipo = 'MÁXIMO' if all(ev < 0) else ('MÍNIMO' if all(ev > 0) else 'SILLA')
print(f'Naturaleza: {tipo}')
yopt = modelo.predict(pd.DataFrame({'x1':[xs[0]], 'x2':[xs[1]]}))[0]
print(f'Rendimiento óptimo predicho: {yopt:.2f}')

Ambos valores propios son **negativos** → el punto estacionario es un **máximo**. El rendimiento se optimiza en ese punto (en variables codificadas).

## 4. Gráfica de contornos

In [ ]:
g = np.linspace(-1.7, 1.7, 60)
X1, X2 = np.meshgrid(g, g)
grid = pd.DataFrame({'x1': X1.ravel(), 'x2': X2.ravel()})
Z = modelo.predict(grid).values.reshape(X1.shape)
fig, ax = plt.subplots(figsize=(6,5))
cs = ax.contourf(X1, X2, Z, levels=15, cmap='viridis')
ax.contour(X1, X2, Z, levels=15, colors='k', linewidths=.3)
ax.plot(xs[0], xs[1], 'r*', ms=16, label='óptimo')
ax.scatter(df.x1, df.x2, c='white', edgecolors='k', s=30, label='corridas')
fig.colorbar(cs, label='rendimiento'); ax.legend()
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Superficie de respuesta (contornos)'); plt.show()

## 5. Conclusiones
- El modelo cuadrático ajusta muy bien ($R^2\approx0.98$).
- El óptimo (un **máximo**) está dentro de la región explorada.
- **Siguiente paso obligado:** corridas de **confirmación** en el óptimo predicho antes de fijar las condiciones de operación.

> Equivalente en R: [`02-rsm-ccd_r.ipynb`](02-rsm-ccd_r.ipynb). En R, el paquete `rsm` automatiza el ajuste y el análisis canónico.